# Brasa — pré-treino do zero (GPU do Colab)
Treina um modelo **próprio, sem pesos externos**, com `python/model.py` do projeto. Escolha *Ambiente de execução → Alterar tipo → GPU* (A100/L4 ideal; T4 serve para o preset `small`).
Tudo é salvo no Drive e **retomável**: se a sessão cair, rode as células de novo.

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/brasa-pretrain'
!mkdir -p {WORK}

In [ ]:
# Repositório privado? Gere um token (GitHub > Settings > Developer settings) e preencha.
GITHUB_TOKEN = ''
import os
url = 'github.com/va415988-glitch/Brasa-IA.git'
auth = f'{GITHUB_TOKEN}@' if GITHUB_TOKEN else ''
if not os.path.exists('/content/Brasa-IA'):
    !git clone https://{auth}{url} /content/Brasa-IA
else:
    !git -C /content/Brasa-IA pull
%cd /content/Brasa-IA
!pip -q install datasets tokenizers numpy

In [ ]:
# 1) Dados abertos + tokenizer próprio. Demora (download). SCALE=1 ~ 7 GB de texto (~2 bi de tokens).
# Para um primeiro teste barato use SCALE=0.1.
SCALE = 1.0
!python pretrain/prepare_data.py --out {WORK}/data --scale {SCALE}

In [ ]:
# 2) Treino. PRESET: 'small' (~40M, T4), 'base' (~100M, A100/L4) ou 'large' (~300M, A100 por dias).
# Se der falta de memória, reduza MICRO. MAX_HOURS deixa o treino parar sozinho antes da sessão acabar.
PRESET, MICRO, MAX_HOURS = 'small', 16, 10
!python pretrain/train.py --data {WORK}/data --out {WORK}/out-{PRESET} --preset {PRESET} --micro-batch {MICRO} --max-hours {MAX_HOURS} --compile

Terminou ou parou por tempo? Rode a célula 2 de novo: ela retoma de `last.pt`.
Pesos finais: `brasa-pretrain/out-<preset>/best.safetensors` (+ `.json`) e o tokenizer em `brasa-pretrain/data/tokenizer.json`. Baixe-os para `model/pretrained/` no seu computador.

## 3) SFT — ensina a conversar
Usa as conversas do projeto + conversas humanas abertas (OpenAssistant, Dolly). Nenhuma resposta vem de outro modelo.
Ao final imprime amostras reais: é o primeiro teste honesto do que o modelo aprendeu.

In [ ]:
!python pretrain/sft.py --data {WORK}/data --base {WORK}/out-{PRESET}/best.safetensors --out {WORK}/sft-{PRESET} --hf oasst1,dolly --epochs 3

Baixe `sft-<preset>/sft.safetensors` (+ `.json`) e `data/tokenizer.json`. No computador: copie o tokenizer para `model/pretrained/tokenizer.json` (caminho que o checkpoint declara) e rode a bancada de programação com o backend `checkpoint`.